![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 15: Interoperability at Scale and Bulk Data



**Health Informatics in Python** · Part IV: Advanced Topics · Module 15 of 16

---



Module 5's FHIR search retrieves one patient at a time — fine for an app, hopeless
for population analytics across millions of records. This module covers
**FHIR Bulk Data** (Flat FHIR), the **asynchronous export** pattern, the **NDJSON**
format it produces, and the network layer: **HIEs** and **TEFCA**.


## Learning objectives

By the end of this module you will be able to:

1. Explain why **bulk export** exists and how it differs from per-resource search.
2. Describe the **asynchronous kickoff → poll → download** Bulk Data flow.
3. Produce and parse **NDJSON** (newline-delimited JSON), the bulk export format.
4. Reassemble bulk NDJSON back into **analytics tables**.
5. Place **HIEs, QHINs, and TEFCA** in the national interoperability picture.


## Dataset

We convert the synthetic EHR into **FHIR NDJSON** exactly as a `$export` would emit
it, then parse it back — simulating both ends of a bulk data exchange locally.


In [1]:
# --- Self-contained synthetic EHR generator (identical to earlier parts) ---
import numpy as np
import pandas as pd

# This function synthesizes a realistic EHR (Electronic Health Record) dataset.
def make_synthetic_ehr(n_patients=200, seed=42):
    # Use a random generator for reproducibility.
    rng = np.random.default_rng(seed)
    # Define pools of first and last names.
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    # Randomly assign demographic details to each patient.
    ages  = rng.integers(18, 90, size=n_patients)
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],          # Unique patient IDs
        "given_name": rng.choice(first, size=n_patients),                 # Given names
        "family_name": rng.choice(last, size=n_patients),                 # Family (last) names
        "sex": rng.choice(["male","female"], size=n_patients, p=[0.49,0.51]), # Biological sex
        "age": ages,                                                      # Age in years
        "birth_year": 2026 - ages,                                        # Calculated birth year (anchor to year 2026)
    })
    # Generate encounter (“visit”) events for each patient.
    enc_rows, enc_types = [], ["ambulatory","emergency","inpatient","wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # Each patient has 1–4 encounters
            day = rng.integers(0, 365*3)  # Encounter within a 3-year range
            enc_rows.append({"encounter_id": f"E{len(enc_rows)+1:05d}", "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55,0.15,0.10,0.20]),  # Most visits are ambulatory
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date()}) # Random date
    encounters = pd.DataFrame(enc_rows)
    # Define common observations (vital signs, labs) and synthesize them for encounters.
    obs_defs = [("Body height","cm",150,195),("Body weight","kg",50,110),
                ("Systolic blood pressure","mmHg",100,165),("Heart rate","/min",55,100),
                ("Hemoglobin A1c","%",4.8,9.5)]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:  # 70% chance of having each observation per encounter
                obs_rows.append({"observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"], "patient_id": e["patient_id"],
                    "observation": name, "value": round(float(rng.uniform(lo,hi)),1),
                    "unit": unit, "date": e["date"]})
    observations = pd.DataFrame(obs_rows)
    # Assign up to 3 random conditions (problems/diagnoses) to each patient.
    cond_pool = ["Essential hypertension","Type 2 diabetes mellitus","Asthma",
                 "Acute bronchitis","Major depressive disorder","Osteoarthritis",
                 "Chronic kidney disease","Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        for c in rng.choice(cond_pool, size=rng.integers(0,4), replace=False):
            cond_rows.append({"condition_id": f"C{len(cond_rows)+1:05d}",
                              "patient_id": pid, "condition": c})
    conditions = pd.DataFrame(cond_rows)
    # Assign up to 3 random medications to each patient.
    med_pool = ["Lisinopril","Metformin","Albuterol","Atorvastatin","Sertraline",
                "Amoxicillin","Ibuprofen","Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        for m in rng.choice(med_pool, size=rng.integers(0,4), replace=False):
            med_rows.append({"medication_id": f"M{len(med_rows)+1:05d}",
                             "patient_id": pid, "medication": m})
    medications = pd.DataFrame(med_rows)
    # Return the synthetic tables as a dictionary keyed by resource type.
    return {"patients":patients,"encounters":encounters,"observations":observations,
            "conditions":conditions,"medications":medications}

# Generate the synthetic EHR dataset (all tables).
ehr = make_synthetic_ehr()
# Print table names and their respective row counts.
print("Tables:", ", ".join(f"{k} ({len(v)})" for k,v in ehr.items()))


Tables: patients (200), encounters (511), observations (1804), conditions (304), medications (276)


## 15.1 Why bulk data

Using per-resource REST (such as `GET /Patient/123`) is highly inefficient for retrieving data about many individuals: pulling a large patient population could require millions of repeated API calls, which is slow and places heavy load on the server.

To solve this, **FHIR Bulk Data** (also known as Flat FHIR) allows exporting entire resource types in a single operation. The server packages data—such as all Patient records, Observations, or Conditions—into compressed **NDJSON** (newline-delimited JSON) files. Each line in these files is a single resource, making them easy to process in parallel or with big data tools.

This capability is foundational for scenarios like population health analysis, clinical quality reporting, and large-scale payer–provider data exchanges, where working with an entire cohort is necessary rather than accessing records one at a time.


## 15.2 The asynchronous export flow

Bulk export in FHIR is designed to be **asynchronous** because exporting large datasets can be resource-intensive and time-consuming, often taking several minutes or more. Rather than tying up server resources and leaving the client waiting for a response, an asynchronous workflow allows efficient handling for both client and server. Here’s how the asynchronous export flow works in detail:

1. **Kickoff** — The client initiates the export by making a request such as `GET [base]/$export`, and includes the `Prefer: respond-async` HTTP header. This header signals to the server that the client is willing to receive an asynchronous response.
2. **Poll for Status** — Instead of immediately returning the full response, the server replies with a `202 Accepted` status and provides a *content-location* header containing the URL where the client can check on the progress of the export job.
3. **Completion and Manifest** — The client periodically polls the content-location endpoint. Once the job has finished, the server responds with a manifest JSON document describing the output. This manifest lists URLs to one or more NDJSON files, typically one file per resource type (e.g., one for all Patient records, one for Observations, etc.).
4. **Download NDJSON Files** — The client then downloads the bulk data by retrieving the NDJSON files from the provided URLs. These files contain the exported data, with each line representing a single resource instance as a JSON object.

In this notebook, we simulate the manifest that a server would return at the completion of a bulk data export job.


In [2]:
import json

# This code simulates a FHIR Bulk Data (Flat FHIR) export manifest,
# which is the response a client receives from the server when a bulk
# data export job completes. The manifest describes the exported data files.

# The manifest is a dictionary containing:
#  - "transactionTime": When the export was completed.
#  - "request": The URL used to initiate the export job.
#  - "requiresAccessToken": Whether the result URLs require authentication.
#  - "output": A list of exported resource files, each specifying:
#       - the resource "type" (e.g., Patient, Observation, Condition)
#       - and the downloadable "url" for each NDJSON file.
#  - "error": Any errors (empty in this simulated manifest).

manifest = {
    "transactionTime": "2026-08-20T00:00:00Z",
    "request": "https://ehr.example/fhir/$export",
    "requiresAccessToken": True,
    "output": [
        {"type": "Patient", "url": "https://ehr.example/bulk/Patient.ndjson"},
        {"type": "Observation", "url": "https://ehr.example/bulk/Observation.ndjson"},
        {"type": "Condition", "url": "https://ehr.example/bulk/Condition.ndjson"},
    ],
    "error": [],
}

# Display the manifest in a readable JSON format
print("Bulk export manifest (returned when the async job completes):")
print(json.dumps(manifest, indent=2))

Bulk export manifest (returned when the async job completes):
{
  "transactionTime": "2026-08-20T00:00:00Z",
  "request": "https://ehr.example/fhir/$export",
  "requiresAccessToken": true,
  "output": [
    {
      "type": "Patient",
      "url": "https://ehr.example/bulk/Patient.ndjson"
    },
    {
      "type": "Observation",
      "url": "https://ehr.example/bulk/Observation.ndjson"
    },
    {
      "type": "Condition",
      "url": "https://ehr.example/bulk/Condition.ndjson"
    }
  ],
  "error": []
}


## 15.3 Producing NDJSON

**NDJSON** stands for Newline Delimited JSON, a format where each line in a text file is a separate JSON object, with no enclosing array or commas between records. 
This approach has several advantages for large-scale data handling:
- **Efficient Streaming:** Clients and servers can read and write data one object at a time without needing to load the entire file into memory.
- **Easy Sharding and Parallelism:** Files can be split and processed in chunks by simply dividing on newline characters.
- **Append-friendly:** New records can easily be appended as new lines.
For these reasons, FHIR bulk export utilizes NDJSON files to export large collections of resources efficiently. 
In the next cell, we demonstrate how to output FHIR `Patient` resources using the NDJSON format, with each patient represented as a separate line.


In [3]:
# Select the "patients" table from the synthetic EHR dataset.
patients = ehr["patients"]

# Define a function to convert each row of the DataFrame
# into a FHIR-compliant Patient resource (dictionary).
def to_fhir_patient(row):
    return {
        "resourceType": "Patient",
        "id": row["patient_id"],
        "name": [{
            "family": row["family_name"],  # Last name
            "given": [row["given_name"]]   # First name as a list
        }],
        "gender": row["sex"],              # HL7/FHIR uses 'gender' field
        "birthDate": f"{row['birth_year']}-01-01"  # Use only the year for privacy
    }

# Take the first 200 patients. For each, serialize the FHIR resource
# as a JSON string (one per line), following NDJSON format.
ndjson_lines = [
    json.dumps(to_fhir_patient(r))
    for _, r in patients.head(200).iterrows()
]

# Join lines together with newline characters to form the NDJSON document.
patient_ndjson = "\n".join(ndjson_lines)

# Display the first 3 NDJSON lines to show the format, and summary stats.
print("Patient.ndjson — first 3 lines:")
for line in ndjson_lines[:3]:
    print(line)
print(f"...\n{len(ndjson_lines)} total resources, {len(patient_ndjson)} bytes")

Patient.ndjson — first 3 lines:
{"resourceType": "Patient", "id": "P1000", "name": [{"family": "Park", "given": ["Sara"]}], "gender": "female", "birthDate": "2002-01-01"}
{"resourceType": "Patient", "id": "P1001", "name": [{"family": "Jain", "given": ["Rex"]}], "gender": "male", "birthDate": "1953-01-01"}
{"resourceType": "Patient", "id": "P1002", "name": [{"family": "Brown", "given": ["Nina"]}], "gender": "female", "birthDate": "1961-01-01"}
...
200 total resources, 27545 bytes


### Milestone 1 - parse NDJSON back into a table

In this milestone, we'll demonstrate how to read a bulk FHIR NDJSON file, where each line is a JSON-encoded resource—and convert it back into a tabular DataFrame. This allows us to round-trip structured FHIR data: exporting it using the NDJSON standard, and then parsing it back for analysis or visualization in Python.

In [4]:
import pandas as pd

# Define a function that takes NDJSON text (each line a JSON-encoded Patient resource)
# and parses it back into a pandas DataFrame for further analysis or visualization.
def parse_patient_ndjson(text):
    rows = []
    # Split the text into lines, strip whitespace, and process each line
    for line in text.strip().split("\n"):
        # Each line is a JSON string representing a FHIR Patient resource
        r = json.loads(line)
        # Extract fields of interest from the FHIR structure and save as a dictionary
        rows.append({
            "id": r["id"],
            "family": r["name"][0]["family"],        # Family name (last name)
            "given": r["name"][0]["given"][0],       # First given name
            "gender": r["gender"],                   # Gender field
            "birthDate": r["birthDate"]              # Birth date
        })
    # Return as a DataFrame for easy manipulation
    return pd.DataFrame(rows)

# Parse the NDJSON (bulk exported) patient data back into a DataFrame
parsed = parse_patient_ndjson(patient_ndjson)

# Display a preview of the round-tripped table and report how many patients were parsed
print("Round-tripped NDJSON -> DataFrame:")
print(parsed.head().to_string(index=False))
print(f"\nParsed {len(parsed)} patients back from bulk NDJSON.")

Round-tripped NDJSON -> DataFrame:
   id family given gender  birthDate
P1000   Park  Sara female 2002-01-01
P1001   Jain   Rex   male 1953-01-01
P1002  Brown  Nina female 1961-01-01
P1003   Park  Theo female 1977-01-01
P1004   Vega  Nina female 1977-01-01

Parsed 200 patients back from bulk NDJSON.


### Milestone 2 - a mini end-to-end bulk client

This section demonstrates a miniature, end-to-end "bulk client" for FHIR data,
showing how to handle downloading, parsing, and performing population-level queries
across large sets of simulated bulk export files.

In [5]:
# --- Explanation of Code for Bulk FHIR Observation Export and Bulk Query ---

# 1. Emit Observations as NDJSON.
# We take the 'observations' table from our simulated EHR data.
obs = ehr["observations"]

# 2. Define key LOINC codes for common observation types.
# LOINC codes standardize lab/measurement concepts in FHIR.
LOINC = {
    "Hemoglobin A1c": "4548-4",
    "Systolic blood pressure": "8480-6",
    "Heart rate": "8867-4",
    "Body weight": "29463-7",
    "Body height": "8302-2"
}

# 3. Serialize each Observation into a line of NDJSON.
# Each line is a FHIR Observation resource in JSON, representing a different clinical measurement.
obs_ndjson = "\n".join(
    json.dumps({
        "resourceType": "Observation",
        "status": "final",
        "code": {
            "coding": [{
                "system": "http://loinc.org",
                "code": LOINC[o["observation"]],
                "display": o["observation"]
            }]
        },
        "subject": {"reference": f"Patient/{o['patient_id']}"},
        "valueQuantity": {"value": float(o["value"]), "unit": o["unit"]}
    })
    for _, o in obs.iterrows()
)

# 4. Simulate Bulk Download
def bulk_download(manifest_output, local_files):
    """
    Given a manifest describing bulk export files, look up each corresponding
    NDJSON string in 'local_files', parse each line as JSON, and
    collect them under their FHIR resource type.
    """
    tables = {}
    for entry in manifest_output:
        text = local_files.get(entry["type"])
        if text:
            # Split NDJSON into a list of JSON objects:
            tables[entry["type"]] = [json.loads(l) for l in text.strip().split("\n")]
    return tables

# 5. Prepare the 'local' exports (Patient and Observation), representing files you might download in bulk.
local = {"Patient": patient_ndjson, "Observation": obs_ndjson}

# 6. Run the simulated bulk download process,
# which will fetch and parse the NDJSON back into Python objects by type.
downloaded = bulk_download(manifest["output"], local)

# Show counts for each resource type downloaded.
print("Downloaded resource counts:", {k: len(v) for k, v in downloaded.items()})

# 7. Perform a population-level query: compute the mean Hemoglobin A1c value.
# This emulates an analytic use-case enabled by bulk export.
a1c_vals = [
    o["valueQuantity"]["value"]
    for o in downloaded.get("Observation", [])
    if o["code"]["coding"][0]["code"] == "4548-4"  # LOINC for HbA1c
]
print(f"Population mean A1c from bulk export: {sum(a1c_vals)/len(a1c_vals):.2f} % "
      f"(n={len(a1c_vals)})")

Downloaded resource counts: {'Patient': 200, 'Observation': 1804}
Population mean A1c from bulk export: 7.10 % (n=356)


## 15.4 The network layer: HIEs, QHINs, and TEFCA

While bulk data export helps move large volumes of health data within a single organization,
exchanging data *across* organizations requires a more advanced network infrastructure.
Here are the main components enabling such nationwide interoperability in the U.S.:

- **HIE (Health Information Exchange)**: These are typically regional networks that facilitate
  the sharing of clinical data (such as lab results, encounters, and medications) among
  multiple healthcare providers, hospitals, and health systems. HIEs help reduce duplication,
  improve care coordination, and enable providers to access patient information regardless
  of where care was delivered.
 
- **TEFCA (Trusted Exchange Framework and Common Agreement)**: TEFCA is a federal initiative
  designed to standardize the rules and technical requirements for health information
  exchange across the entire country. It provides a "network of networks" by creating a
  shared trust framework, data sharing policies, and technical standards, ensuring that all
  participants can rely on common ground rules and interoperability.

- **QHIN (Qualified Health Information Network)**: QHINs are large, certified networks that
  connect directly to TEFCA. They serve as hubs that link together different HIEs, health
  systems, payers, and other qualified entities. Once connected, a data request made in one
  region or network can securely reach and retrieve data from other regions through the mesh
  created by these QHINs.

**Direction of travel:** With the adoption of modern standards like FHIR and the national
trust framework of TEFCA, exchanging health information across the U.S. is moving from
custom-built solutions to standardized, routine interoperability at scale.


## Exercises

1. Add a `Condition` NDJSON stream and extend `bulk_download` to compute condition
   prevalence across the export.
2. Implement a **`_since` parameter**: filter the export to resources after a given
   date (incremental/delta export).
3. Gzip the NDJSON (`gzip` module) and report the compression ratio — bulk files
   are always compressed in practice.

## Key takeaways

- **Bulk Data / Flat FHIR** exports whole populations as **NDJSON**, unlike chatty
  per-resource search.
- The flow is **async**: kickoff → poll → manifest → download.
- **NDJSON** streams and shards; parsing it back yields analytics tables.
- **HIEs, QHINs, and TEFCA** carry exchange *across* organizations nationwide.

---
*Next: Module 16 - Emerging Directions.*
